In [1]:
# Cell 1.0 — fix proxy for this notebook only (does not change system settings)
import os

for k in ["ALL_PROXY", "all_proxy"]:
    os.environ.pop(k, None)

HTTP_PROXY = "http://127.0.0.1:10808"   # v2ray mixed port (HTTP + SOCKS)
for k in ["HTTP_PROXY", "http_proxy", "HTTPS_PROXY", "https_proxy"]:
    os.environ[k] = HTTP_PROXY

print({k: os.environ.get(k) for k in ["ALL_PROXY", "HTTP_PROXY", "HTTPS_PROXY"]})

{'ALL_PROXY': None, 'HTTP_PROXY': 'http://127.0.0.1:10808', 'HTTPS_PROXY': 'http://127.0.0.1:10808'}


In [2]:
import sys, platform, datasets, pandas as pd
from datasets import load_dataset

In [3]:
print("Python:", sys.version.split()[0], "| OS:", platform.platform())

Python: 3.14.4 | OS: Linux-7.0.0-34-generic-x86_64-with-glibc2.43


In [4]:
print("datasets:", datasets.__version__, "| pandas:", pd.__version__)

datasets: 5.1.0 | pandas: 3.0.6


In [5]:
CONFIGS = ["wix_kb_corpus", "wixqa_synthetic", "wixqa_expertwritten", "wixqa_simulated"]
ds = {}
for name in CONFIGS:
    ds[name] = load_dataset("Wix/WixQA", name, revision="d662dc42479c14e202eccd832f8c4b66a035c4cc")
    print(f"\n=== {name} ===")
    for split, d in ds[name].items():
        print(f"split='{split}'  rows={d.num_rows}")
        print(d.features)


=== wix_kb_corpus ===
split='train'  rows=6221
{'id': Value('string'), 'url': Value('string'), 'contents': Value('string'), 'title': Value('string'), 'html_content': Value('string'), 'article_type': Value('string')}

=== wixqa_synthetic ===
split='train'  rows=6221
{'question': Value('string'), 'answer': Value('string'), 'article_ids': List(Value('string'))}

=== wixqa_expertwritten ===
split='train'  rows=200
{'question': Value('string'), 'answer': Value('string'), 'article_ids': List(Value('string'))}

=== wixqa_simulated ===
split='train'  rows=200
{'question': Value('string'), 'answer': Value('string'), 'article_ids': List(Value('string'))}


In [6]:
# Cell 1.2 — record dataset revision + convert to pandas
from huggingface_hub import HfApi
print("WixQA revision (commit sha):", HfApi().dataset_info("Wix/WixQA").sha)

kb = ds["wix_kb_corpus"]["train"].to_pandas()
qa = {name: ds[name]["train"].to_pandas()
      for name in ["wixqa_synthetic", "wixqa_expertwritten", "wixqa_simulated"]}
kb_by_id = kb.set_index("id")

print("KB ids unique:", kb["id"].is_unique)
print({name: df.shape for name, df in qa.items()}, "| KB:", kb.shape)

WixQA revision (commit sha): d662dc42479c14e202eccd832f8c4b66a035c4cc
KB ids unique: True
{'wixqa_synthetic': (6221, 3), 'wixqa_expertwritten': (200, 3), 'wixqa_simulated': (200, 3)} | KB: (6221, 6)


In [7]:
# Cell 1.3 — inspect examples: question, answer, and titles of gold articles
SEED = 42

def show_examples(name, n=10, width=300):
    sample = qa[name].sample(n, random_state=SEED)
    for i, row in sample.iterrows():
        print(f"\n--- [{name}] row {i} ---")
        print("Q:", row["question"][:width])
        print("A:", row["answer"][:width].replace("\n", " "))
        for aid in row["article_ids"]:
            title = kb_by_id["title"].get(aid, "<<NOT IN KB>>")
            print(f"   gold id={aid} | title={title}")

show_examples("wixqa_synthetic")


--- [wixqa_synthetic] row 2107 ---
Q: How can I change the color of the drop-down arrow on the mobile menu in Wix Editor?
A: To change the color of the drop-down arrow on the mobile menu in Wix Editor, follow these steps:  1. Go to your [mobile editor](https://support.wix.com/en/article/wix-editor-getting-started-with-the-mobile-editor). 2. Click the mobile menu. 3. Click **Edit Menu**. 4. Click the menu. 5. Click the **D
   gold id=9753b78fbde38779268be1488d16210afe81aca489a066289a78f1fdad7329ef | title=Wix Editor: Changing the Color of the Drop-Down Arrow on the Mobile Menu

--- [wixqa_synthetic] row 3813 ---
Q: How can I assign a new domain to my Wix site?
A: Prerequisites: You must upgrade your site with a plan before you can assign a domain to it.  To assign a new domain to your Wix site, follow these steps:  1. [Go to Domains](https://manage.wix.com/wix/api/mpContainerStaticController#/domains) in your Wix account. 2. Assign your domain to your site: 
   gold id=0517ef7fc15c2473

In [8]:
show_examples("wixqa_expertwritten")


--- [wixqa_expertwritten] row 95 ---
Q: Im unable to access the Google Maps element in the footer of my Wix site. Im having trouble removing it, and I dont know where the overlapping items feature is located.
A: If you are unable to select or find an element in the Editor, it is possible that it has been placed behind another element.  If an element is placed outside the Editor gridlines or behind another element, you may not be able to see it on your screen. Use the Layers panel to locate the element on th
   gold id=be8f488cde5a1ed4c46d21398144c082d689a03c700094b17d1c9ebb1ee6902e | title=Wix Editor: Unable to Select an Element

--- [wixqa_expertwritten] row 15 ---
Q: I need help managing the categories my booking services are in on my Wix website.
A: To manage the categories for your booking services on your Wix website, you can follow these steps:  **Creating categories** 1. Go to Booking Services in your site's dashboard. 2. Click Manage Categories. 3. Click + Add New Category. 4.

In [9]:
show_examples("wixqa_simulated")


--- [wixqa_simulated] row 95 ---
Q: my customer wants to buy a gift card and other products as one purchase. how do they do that?
A: Currently, it's not possible to add gift cards and other products to the cart and pay for them together. Customers need to purchase Wix Gift Cards separately from other products. As a workaround, you can use the Gifted app, a third-party solution that integrates with Wix Stores, allowing customers t
   gold id=4a2853d7fa4660c1214cc9ead8ade3c206435e28e89aab97ec577f78768a0597 | title=Wix Gift Cards Request: Purchasing Gift Cards with Other Products

--- [wixqa_simulated] row 15 ---
Q: how to take my site off live
A: To take your Wix site offline, you can unpublish it by opening your site in the Wix Editor, selecting 'Site' in the top toolbar, clicking 'Unpublish Site' and confirming. Alternatively, you can unpublish from your sites dashboard by going to Settings > Website settings > Unpublish.
   gold id=0cb65c6c64d2b1d3c54f04b08cd7c801334d1984fc5f07e6e139

In [10]:
# Cell 1.4 — integrity checks: empty fields, gold ids per question, gold ids present in KB
kb_ids = set(kb["id"])

def is_empty(s):
    return s.isna() | (s.astype(str).str.strip() == "")

print("KB empty fields:", {c: int(is_empty(kb[c]).sum()) for c in kb.columns})
print("KB article_type counts:", kb["article_type"].value_counts(dropna=False).to_dict())

for name, df in qa.items():
    n_ids = df["article_ids"].apply(len)
    all_ids = [a for ids in df["article_ids"] for a in ids]
    missing = [a for a in all_ids if a not in kb_ids]
    print(f"\n=== {name} ===")
    print("empty question / answer:", int(is_empty(df["question"]).sum()), "/", int(is_empty(df["answer"]).sum()))
    print("duplicate questions:", int(df["question"].duplicated().sum()))
    print("gold ids per question:", n_ids.value_counts().sort_index().to_dict())
    print("gold ids not found in KB:", len(missing), "of", len(all_ids))
    print("distinct gold articles:", len(set(all_ids)))

KB empty fields: {'id': 0, 'url': 0, 'contents': 0, 'title': 0, 'html_content': 0, 'article_type': 0}
KB article_type counts: {'article': 4109, 'feature_request': 2049, 'known_issue': 63}

=== wixqa_synthetic ===
empty question / answer: 0 / 0
duplicate questions: 0
gold ids per question: {1: 6221}
gold ids not found in KB: 0 of 6221
distinct gold articles: 6221

=== wixqa_expertwritten ===
empty question / answer: 0 / 0
duplicate questions: 0
gold ids per question: {1: 148, 2: 46, 3: 6}
gold ids not found in KB: 0 of 258
distinct gold articles: 209

=== wixqa_simulated ===
empty question / answer: 0 / 0
duplicate questions: 0
gold ids per question: {1: 173, 2: 24, 3: 3}
gold ids not found in KB: 0 of 230
distinct gold articles: 160


In [11]:
# Cell 1.5 — length statistics (whitespace words) + results/dataset_summary.csv
from pathlib import Path

def words(s):
    return s.str.split().str.len()

def stats(x, prefix):
    return {f"{prefix}_mean": round(x.mean(), 1), f"{prefix}_median": x.median(),
            f"{prefix}_p95": x.quantile(0.95), f"{prefix}_max": x.max()}

rows = []
for name, df in qa.items():
    n_ids = df["article_ids"].apply(len)
    row = {"subset": name, "n_rows": len(df), "n_gold_ids": int(n_ids.sum()),
           "n_distinct_gold": len({a for ids in df["article_ids"] for a in ids}),
           "pct_multi_gold": round((n_ids > 1).mean() * 100, 1)}
    row.update(stats(words(df["question"]), "q_words"))
    row.update(stats(words(df["answer"]), "a_words"))
    rows.append(row)

kb_words = words(kb["contents"])
kb_row = {"subset": "wix_kb_corpus", "n_rows": len(kb)}
kb_row.update(stats(kb_words, "doc_words"))
rows.append(kb_row)

summary = pd.DataFrame(rows)
Path("../results").mkdir(exist_ok=True)          # notebook lives in notebooks/
summary.to_csv("../results/dataset_summary.csv", index=False)
print(summary.T)

print("\nKB doc words by article_type:")
print(kb.assign(w=kb_words).groupby("article_type")["w"].describe(percentiles=[.5, .95]).round(0))
print("\nKB docs with < 30 words:", int((kb_words < 30).sum()))

                                0                    1                2  \
subset            wixqa_synthetic  wixqa_expertwritten  wixqa_simulated   
n_rows                       6221                  200              200   
n_gold_ids                 6221.0                258.0            230.0   
n_distinct_gold            6221.0                209.0            160.0   
pct_multi_gold                0.0                 26.0             13.5   
q_words_mean                 14.1                 19.9             12.3   
q_words_median               14.0                 18.0             12.0   
q_words_p95                  20.0                 41.0             20.0   
q_words_max                  35.0                 62.0             37.0   
a_words_mean                104.9                149.5             43.7   
a_words_median               75.0                121.5             44.0   
a_words_p95                 306.0               383.05             71.0   
a_words_max              

In [12]:
# Cell 1.6 — inspect KB articles: 10 random + 5 shortest
def show_kb(df, width=400):
    for _, r in df.iterrows():
        w = len(r["contents"].split())
        print(f"\n--- id={r['id'][:12]}… | type={r['article_type']} | words={w}")
        print("TITLE:", r["title"])
        print("TEXT :", r["contents"][:width].replace("\n", " "))

show_kb(kb.sample(10, random_state=SEED))
print("\n\n===== 5 shortest KB docs =====")
show_kb(kb.loc[kb_words.nsmallest(5).index])


--- id=1a119f28f47a… | type=article | words=439
TITLE: Wix Editor: Creating a Site Menu with Anchors and Sections for One-Page Sites
TEXT : Wix Editor: Creating a Site Menu with Anchors and Sections for One-Page Sites Even though one-page sites cannot have a regular site menu (with tabs that link to different pages), you can create a site menu with anchor and section links instead. You can place anchors and sections wherever you like on your one-page site, and when your visitors click the links on your menu, they are directed to the r

--- id=563055de4af7… | type=article | words=277
TITLE: Wix Editor: Adding and Setting Up a YouTube Subscribe Button
TEXT : Wix Editor: Adding and Setting Up a YouTube Subscribe Button Adding a Youtube subscribe button gives your visitors an easy way to follow your channel without ever having to leave your site. Once visitors click the button, they can log in to their YouTube account and confirm the subscription.Step 1 | Retrieve Your YouTube Channel URL

In [13]:
# Cell 1.7 — does contents start with title? how many docs are title-only?
c = kb["contents"].str.strip()
t = kb["title"].str.strip()
starts_with_title = pd.Series([ci.startswith(ti) for ci, ti in zip(c, t)], index=kb.index)
body = pd.Series([ci[len(ti):].strip() if s else ci for ci, ti, s in zip(c, t, starts_with_title)], index=kb.index)
title_only = body.str.len() == 0

print("contents starts with title:", int(starts_with_title.sum()), "of", len(kb))
print("title-only docs:", int(title_only.sum()), kb.loc[title_only, "article_type"].value_counts().to_dict())

to_ids = set(kb.loc[title_only, "id"])
for name, df in qa.items():
    n = df["article_ids"].apply(lambda ids: any(a in to_ids for a in ids)).sum()
    print(f"{name}: questions whose gold includes a title-only doc = {int(n)}")

contents starts with title: 6221 of 6221
title-only docs: 29 {'known_issue': 28, 'article': 1}
wixqa_synthetic: questions whose gold includes a title-only doc = 29
wixqa_expertwritten: questions whose gold includes a title-only doc = 0
wixqa_simulated: questions whose gold includes a title-only doc = 0
